# Lab 4 : Should We Run the Promotion?

Use Monte Carlo experiments to compare profit and risk, then decide whether the matcha promotion is worth launching.

Complete the code TODOs, the short written responses, manager recommendation, and AI-use statement. Think about the discussion questions and be prepared to explain your reasoning to the TA. Every code location that you must complete is marked `TODO`; code marked **Provided** or **do not edit** should be left unchanged.

This notebook contains both the practical work and the report. Submit only the completed notebook.

## Student Information

**Name:** Write your answer here  
**Student ID:** Write your answer here  
**Date:** Write your answer here

## Setup

Run the imports before starting Part A. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Part A: How Much Can You Trust One Simulated Number?

Today you hand the café manager a single number, the estimated daily profit of running the matcha promotion and she may spend real money on the strength of it. So before simulating the café at all, find out how much a Monte Carlo number moves around when you run the same experiment again.

We can write an integral as an expectation:

$$
I=\int_0^1 x^2\,dx=\mathbb{E}[U^2],
\qquad U\sim\operatorname{Uniform}(0,1),
\qquad \widehat I_n=\frac{1}{n}\sum_{i=1}^{n}U_i^2 ,
$$

and the exact value is $1/3$. Three things to establish here, each of which you will use on the café later:

- **A1:** Does the estimate settle down as you simulate more? *(Law of Large Numbers)*
- **A2:** How much does the answer change if you run the whole thing again? *(Central Limit Theorem)*
- **A3:** What does "95% confident" actually promise? *(confidence interval)*

### A1: Does the Estimate Settle Down?

Create one generator with seed `211`, then continue its stream for sample sizes 10, 100, 1,000, and 10,000.

In [ ]:
rng = np.random.default_rng(211)
exact_integral = 1 / 3
sample_sizes = [10, 100, 1000, 10000]
integration_rows = []

for n in sample_sizes:
    # Generate n independent Uniform(0, 1) values.
    u = TODO

    # Calculate the Monte Carlo estimate and its absolute error.
    estimate = TODO
    absolute_error = TODO

    # Provided table-row code -- do not edit.
    integration_rows.append({
        "n": n,
        "estimate": estimate,
        "absolute_error": absolute_error,
    })

# Provided table code -- do not edit.
integration_table = pd.DataFrame(integration_rows)
integration_table

In [ ]:
# Provided plotting code -- do not edit.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(integration_table["n"], integration_table["estimate"], marker="o", label="Estimate")
ax.axhline(exact_integral, color="black", linestyle="--", label="Exact value")
ax.set_xscale("log")
ax.set_xlabel("Sample size n")
ax.set_ylabel("Monte Carlo estimate")
ax.set_title("Monte Carlo estimate of the integral of x²")
ax.legend()
plt.show()

**A1 questions.** Think about these.

1. Does the estimate become exactly equal to `1/3`?
2. Must the absolute error decrease at every larger sample size? If not, why not?

### A2: Run It Again and You Get a Different Answer

A1 gave **one** estimate for each $n$. Run the same experiment with fresh random numbers and you get a different number which is exactly what worries a manager being handed a single figure. The Central Limit Theorem describes how much those numbers scatter:

$$
\widehat I_n\;\approx\;\operatorname{Normal}\!\left(I,\;\frac{\sigma_h^2}{n}\right),
\qquad \sigma_h^2=\operatorname{Var}\big(h(U)\big).
$$

Here $h(U)=U^2$, and this variance can be computed exactly:

$$
\sigma_h^2=\mathbb{E}[U^4]-\big(\mathbb{E}[U^2]\big)^2=\tfrac15-\tfrac19=\tfrac{4}{45},
\qquad \sigma_h=\sqrt{4/45}\approx0.29814 .
$$

That makes the CLT a **prediction you can check**: 200 repeated estimates from samples of size $n=100$ should scatter with standard deviation about $0.29814/\sqrt{100}=0.02981$, and at $n=400$ about $0.01491$, half as much, because the sample is four times bigger.

Complete the three TODOs below, then see whether the prediction holds.

In [ ]:
def repeated_estimates(n, repeats, rng, z=1.96):
    """Repeat the Part A experiment `repeats` times and record each result.

    One row per repeated experiment: the estimate, the sample standard
    deviation of the outputs h(U_i), the standard error, and a 95% interval.
    """
    rows = []

    for _ in range(repeats):
        u = rng.random(n)
        h = u ** 2                      # provided: the outputs h(U_i)

        # TODO: the Monte Carlo estimate for this repetition.
        estimate = TODO

        # TODO: the sample standard deviation of h (use ddof=1).
        sample_sd = TODO

        # TODO: the standard error of the estimate.
        standard_error = TODO

        rows.append({
            "estimate": estimate,
            "sample_sd": sample_sd,
            "standard_error": standard_error,
            "lower": estimate - z * standard_error,
            "upper": estimate + z * standard_error,
        })

    return pd.DataFrame(rows)

In [ ]:
# Provided -- DO NOT EDIT.
clt_rng = np.random.default_rng(2110)
sigma_h = np.sqrt(4 / 45)

repeats_100 = repeated_estimates(n=100, repeats=200, rng=clt_rng)
repeats_400 = repeated_estimates(n=400, repeats=200, rng=clt_rng)

clt_check = pd.DataFrame([
    {
        "n": n,
        "mean of 200 estimates": frame["estimate"].mean(),
        "SD of 200 estimates": frame["estimate"].std(ddof=1),
        "CLT prediction sigma_h/sqrt(n)": sigma_h / np.sqrt(n),
        "mean reported standard error": frame["standard_error"].mean(),
    }
    for n, frame in [(100, repeats_100), (400, repeats_400)]
])
clt_check.round(5)

In [ ]:
# Provided plotting code -- DO NOT EDIT.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(repeats_100["estimate"], bins=25, density=True, alpha=0.55,
        edgecolor="white", label="200 estimates, n = 100")

grid = np.linspace(repeats_100["estimate"].min(), repeats_100["estimate"].max(), 300)
predicted_sd = sigma_h / np.sqrt(100)
normal_density = np.exp(-0.5 * ((grid - exact_integral) / predicted_sd) ** 2) / (
    predicted_sd * np.sqrt(2 * np.pi)
)
ax.plot(grid, normal_density, color="black", label="CLT prediction")
ax.axvline(exact_integral, color="black", linestyle="--", label="Exact value 1/3")
ax.set_xlabel("Monte Carlo estimate")
ax.set_ylabel("Density")
ax.set_title("Sampling distribution of the estimator")
ax.legend()
plt.show()

**A2 response**

Compare the observed SD of the 200 estimates with the CLT prediction at each $n$, and say by what factor the SD changed when $n$ went from 100 to 400 against the factor the CLT predicts.

**Write two or three sentences here.**

*Also think about*, A1 produced a single estimate at $n=100$. Where does that one number sit in the histogram above?

### A3: What "95% Confident" Actually Promises

Every simulation report you will ever read quotes an interval. This is where you find out what it is claiming:

$$
\widehat I_n\pm1.96\frac{s_h}{\sqrt n},
$$

with the **sample** standard deviation, because in a real problem — the café included — $\sigma_h$ is unknown. `repeated_estimates` already stored an interval for each of the 200 repetitions, so the claim can be tested directly: if it is honest, about 95% of those intervals should contain $1/3$.

In [ ]:
# Provided -- DO NOT EDIT.
coverage_rows = []
for n, frame in [(100, repeats_100), (400, repeats_400)]:
    contains_truth = (frame["lower"] <= exact_integral) & (exact_integral <= frame["upper"])
    coverage_rows.append({
        "n": n,
        "intervals containing 1/3": int(contains_truth.sum()),
        "out of": len(frame),
        "empirical coverage": contains_truth.mean(),
    })

coverage_table = pd.DataFrame(coverage_rows)
coverage_table

**A3 response.**

State the empirical coverage you obtained and whether it is close to 95%, then say which of the two is random: the interval, or the value $1/3$.

**Write one or two sentences here.**

*Also think about:* for the café you will compute **one** interval, for a profit whose true value nobody knows. What does the 95% refer to then? 

---

*You now know that a Monte Carlo answer settles down slowly, scatters like a Normal with spread $\sigma_h/\sqrt n$, and comes with an interval that is right about 95% of the time. Back to the café.*

## Part B: Define the Decision

The coffee-shop manager is considering a matcha promotion. It may raise demand and customer spending, but it also has a campaign cost and may make profit less predictable.

Two things to settle before you simulate anything. The judgement whether the promotion is worth running, and what would make you reject it belongs in Part G, once you have the numbers.

**Part B response** What counts as a **bad day** here, and why must that be stated before simulating rather than after? Which model inputs are **uncertain**, and which are fixed decisions or known costs?

**Write two or three sentences here.**

## Part C: The Provided Daily-Profit Model

The two assumptions dictionaries and the function `simulate_day_profit` are **provided and must not be edited**. Your job in this part is to read the model, run it, and be able to explain it.

For one day, with demand $D$ and individual customer spends $V_1,\dots,V_D$,

$$
\text{profit}=\sum_{i=1}^{D}V_i-cD-F-K,
$$

where $c$ is the variable cost per customer, $F$ the fixed daily cost, and $K$ the promotion cost. Both the number of terms and the terms themselves are random, so the function draws demand first and then draws exactly that many spending values.

Demand is a rounded Normal value **clipped** at zero, and each customer spend is a Normal value clipped at zero. Clipping replaces a negative draw by zero; it is not the same as a truncated Normal, which would condition on the value being positive. Here the largest probability of a negative draw is about $1.0\times10^{-6}$, so the distortion is negligible in size but the wording still matters.

In [ ]:
# Provided assumptions -- DO NOT EDIT.
baseline = {
    "demand_mean": 100,
    "demand_sd": 15,
    "spend_mean": 5.5,
    "spend_sd": 1.0,
    "variable_cost": 2.4,
    "fixed_cost": 250,
    "promotion_cost": 0,
}

promotion = {
    "demand_mean": 125,
    "demand_sd": 25,
    "spend_mean": 5.7,
    "spend_sd": 1.2,
    "variable_cost": 2.6,
    "fixed_cost": 250,
    "promotion_cost": 70,
}

In [ ]:
# Provided model -- DO NOT EDIT.
def simulate_day_profit(assumptions, rng):
    """Return the simulated profit of one day.

    assumptions : dict of model parameters
    rng         : a NumPy Generator, created OUTSIDE this function
    """
    # Number of customers: a rounded Normal draw, clipped at 0 and made an integer
    # because it is used as the number of spending values to draw.
    demand = max(0, int(round(rng.normal(assumptions["demand_mean"],
                                        assumptions["demand_sd"]))))

    # One spending value per customer: an array of length `demand`, clipped at 0.
    customer_spend = np.maximum(0, rng.normal(assumptions["spend_mean"],
                                              assumptions["spend_sd"],
                                              size=demand))

    revenue = customer_spend.sum()
    total_cost = (assumptions["variable_cost"] * demand
                  + assumptions["fixed_cost"]
                  + assumptions["promotion_cost"])

    return revenue - total_cost

In [ ]:
# Provided check -- DO NOT EDIT. Simulate five days for each case.
rng = np.random.default_rng(211)

baseline_profits = [simulate_day_profit(baseline, rng) for _ in range(5)]
promotion_profits = [simulate_day_profit(promotion, rng) for _ in range(5)]

one_day_checks = pd.DataFrame({
    "Baseline profit": baseline_profits,
    "Promotion profit": promotion_profits,
})
one_day_checks

### Reading the Model

Run the two cells above and confirm that daily profit changes from day to day.

1. Why does the spending draw use `size=demand`, and why must `demand` be an integer?
2. Why is `rng` passed into the function instead of being created inside it?
3. Which quantities in the model are random, and which are fixed?

### Controlled Promotion-Cost Check

Compare promotion costs 0, 70, and 300. The check seed is recreated before each cost, so every run draws the same demand and the same spending values and only the cost differs.

In [ ]:
check_seed = 212
costs_to_check = [0, 70, 300]
cost_check_profits = []

for promotion_cost in costs_to_check:
    changed_promotion = promotion.copy()
    changed_promotion["promotion_cost"] = promotion_cost
    check_rng = np.random.default_rng(check_seed)

    # Simulate the promotion once with identical random inputs for every cost.
    checked_profit = TODO
    cost_check_profits.append(checked_profit)

# Provided table code -- do not edit.
promotion_cost_checks = pd.DataFrame({
    "Promotion cost": costs_to_check,
    "Profit with identical random inputs": cost_check_profits,
})
promotion_cost_checks

**Question**

What exactly changed between the three rows, and by how much? Why does reusing the same check seed make this a controlled comparison?

**Write one or two sentences here.**

*Also think about:* suppose you changed `demand_mean` instead of `promotion_cost`, keeping the same seed. Would the two runs still use identical random inputs, and what happens to the stream of random numbers? 

## Part D: Run Many Replications

For simulated day $i$, let $X_i$ contain its random demand and spending inputs, and let $h(X_i)$ be daily profit. The mean of many values $h(X_i)$ estimates $\mathbb{E}[h(X)]$, the expected daily profit.

Use seed `211` and run 1,000 replications for each case.

In [ ]:
def run_profit_experiment(n_replications, seed=211):
    rng = np.random.default_rng(seed)
    rows = []

    for case, assumptions in [("Baseline", baseline), ("Promotion", promotion)]:
        for replication in range(1, n_replications + 1):
            # Simulate one profit and store one long-format row.
            profit = TODO
            rows.append({
                "replication": replication,
                "case": case,
                "profit": profit,
            })

    return pd.DataFrame(rows)


# Run the required experiment.
profit_df = TODO

# Provided checks -- do not edit.
assert len(profit_df) == 2000, "Expected 1,000 rows for each of two cases."
assert set(profit_df.columns) == {"replication", "case", "profit"}
profit_df.head()

In [ ]:
# Provided summary-table code -- do not edit.
profit_summary = profit_df.groupby("case")["profit"].agg(
    mean_profit="mean",
    standard_deviation="std",
    minimum="min",
    maximum="max",
)
profit_summary.round(4)

### Part D Questions

1. Why are the two standard deviations different?
2. Why should the generator not be reset inside `simulate_day_profit`?
3. Can two cases have similar means but different risks?
4. The loop runs all baseline days first and then all promotion days from the same stream, so the two cases use *different* random numbers. If both cases instead reused the same random inputs day by day, would the estimated **difference** in mean profit become more or less noisy?

## Checkpoint 1 (1 of 3 Points)

Show the TA your completed Parts A-D: the A1 integration table and plot, the A2 CLT comparison and histogram, the A3 coverage table, and your four written responses so far (A2, A3, Part B, controlled check), plus the 2,000-row replication DataFrame with its profit summary.

## Part E: Bad-Day Probability

Define $Y_i=\mathbf{1}\{\text{profit}_i<0\}$. The average of these zero-one indicators estimates the probability of a bad day.

In [ ]:
bad_day_threshold = 0

# Create one Boolean bad-day indicator for every replication.
bad_day_indicator = TODO

# Provided probability-table code -- do not edit.
bad_day_table = (
    profit_df.assign(bad_day=bad_day_indicator)
    .groupby("case")["bad_day"]
    .mean()
    .rename("bad_day_probability")
    .to_frame()
)
bad_day_table

**Part E question.** The promotion raises the estimated mean profit *and* the estimated bad-day probability. Which of those two numbers would a manager with limited cash care about more, and why? 

## Part F: Applying the Interval, and the Convergence Rate

Part A3 established what a 95% interval means and checked it against a known truth. Here you apply the same formula to a quantity whose true value you do **not** know:

$$
\bar h_n\pm1.96\frac{s_h}{\sqrt{n}} .
$$

The function below is the same calculation you wrote in Part A2, so it is provided rather than repeated. Remember that the interval describes Monte Carlo uncertainty in the **mean profit**, not the range of individual daily profits.

In [ ]:
# Provided -- do not edit. Same calculation as in Part A2.
def mean_confidence_interval(values, z=1.96):
    values = np.asarray(values)
    n = len(values)
    mean = values.mean()
    sample_sd = values.std(ddof=1)
    standard_error = sample_sd / np.sqrt(n)
    lower = mean - z * standard_error
    upper = mean + z * standard_error
    return mean, lower, upper, upper - lower


# Provided application code -- do not edit.
ci_rows = []
for case, group in profit_df.groupby("case"):
    mean, lower, upper, width = mean_confidence_interval(group["profit"])
    ci_rows.append({
        "case": case,
        "mean": mean,
        "lower": lower,
        "upper": upper,
        "ci_width": width,
    })

ci_table = pd.DataFrame(ci_rows).set_index("case")
ci_table.round(4)

In [ ]:
stability_rows = []

for n_replications in [100, 1000, 10000]:
    # Reset to seed 211 at the start of each complete experiment.
    experiment = TODO

    # Provided summarising code -- do not edit.
    for case, group in experiment.groupby("case"):
        mean, lower, upper, width = mean_confidence_interval(group["profit"])
        stability_rows.append({
            "replications": n_replications,
            "case": case,
            "mean": mean,
            "ci_width": width,
        })

stability_long = pd.DataFrame(stability_rows)
stability_table = stability_long.pivot(
    index="replications", columns="case", values=["mean", "ci_width"]
)
stability_table.round(4)

**What the stability table is and is not.** Each experiment restarts seed `211` and runs the baseline case first, so the 100 baseline days are exactly the first 100 of the 1,000 baseline days. The rows are a *nested* view of one stream stabilising, not three independent experiments. That is what you want for seeing the Law of Large Numbers, but the three estimates are not independent checks of each other.

In [ ]:
# Provided plotting code -- do not edit.
fig, ax = plt.subplots(figsize=(8, 4))
for case, group in profit_df.groupby("case"):
    ax.hist(group["profit"], bins=25, alpha=0.55, label=case)
ax.axvline(0, color="black", linestyle="--", label="Bad-day threshold")
ax.set_xlabel("Daily profit")
ax.set_ylabel("Frequency")
ax.set_title("Monte Carlo profit distributions")
ax.legend()
plt.show()

### Does Any of This Apply to the Café?

Part A2 showed the bell shape for a toy integral. The manager is entitled to ask whether it says anything about her shop. So run the promotion experiment — all 1,000 replications of it — 200 times over, and look at the 200 answers you could have reported.

This takes a couple of seconds to run.

In [ ]:
# Provided -- do not edit. 200 repeats of the full 1,000-replication promotion experiment.
repeat_rng = np.random.default_rng(211)
promotion_means = np.array([
    np.mean([simulate_day_profit(promotion, repeat_rng) for _ in range(1000)])
    for _ in range(200)
])

promotion_sd = profit_df.loc[profit_df["case"] == "Promotion", "profit"].std(ddof=1)

pd.Series({
    "mean of the 200 reported means": promotion_means.mean(),
    "SD of the 200 reported means": promotion_means.std(ddof=1),
    "CLT prediction s_h / sqrt(1000)": promotion_sd / np.sqrt(1000),
}).round(4)

In [ ]:
# Provided plotting code -- do not edit.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(promotion_means, bins=20, alpha=0.6, edgecolor="white")
ax.axvline(ci_table.loc["Promotion", "mean"], color="black", linestyle="--",
           label="the mean you reported in Part F")
ax.set_xlabel("Mean daily profit under the promotion")
ax.set_ylabel("Frequency")
ax.set_title("200 answers you could have reported")
ax.legend()
plt.show()

**Café response.** This is the one the manager actually needs.

The histogram just above, of *individual daily profits*, is wide and skewed; this one, of *mean* daily profit, is narrow and roughly symmetric. Both describe the same promotion. Explain the difference to the manager in one sentence, then say whether the spread of the 200 reported means is close to the CLT prediction $s_h/\sqrt{1000}$.

**Write two or three sentences here.**

*Also think about:* the mean you reported in Part F is marked on the histogram. Was it a typical answer, or an unusual one?

### The Convergence Rate

The Monte Carlo error is $O(n^{-1/2})$: multiplying the number of replications by 100 should divide the interval width by about 10. Check it on your own table.

**Rate response — one box.** Divide the CI width at 100 replications by the width at 10,000, for each case. What two ratios do you get, and why are they not exactly 10?

**Write one or two sentences here.**

*Also think about:* does the estimated mean become perfectly stable as $n$ grows, and can a narrow interval rescue an unrealistic demand or cost assumption?

## Part G: Recommendation and AI-Use Statement

Write a short recommendation that states a decision and uses estimated mean profit, bad-day probability, the confidence intervals, and at least one model limitation.

**Manager recommendation:** Write your answer here.

State whether AI tools were used. If they were used, identify the tool, explain how it was used, and state how you checked, corrected, or revised its output. You must be able to explain all submitted work.

**AI-use statement:** Write your answer here.

## Checkpoint 2: About Minute 105 — Final Submission

1. Show the completed Parts E-G, recommendation, and AI-use statement to the TA.
2. Answer one short code question and one modelling question.
3. Save the notebook as `Lab04_StudentID.ipynb`, for example `Lab04_6812345.ipynb`.
4. Restart the kernel and run all cells from top to bottom.
5. Check that there are no errors, save, and upload the single completed notebook to the **Lab 04** assignment in Google Classroom.

## Optional Extension: Variance Reduction

Attempt this only after Checkpoint 2. It is not assessed.

In Part A2 you measured the spread of 200 Monte Carlo estimates and found it close to $\sigma_h/\sqrt n$. Variance reduction changes $\sigma_h$ itself rather than $n$. The antithetic estimator pairs each $U$ with $1-U$:

$$
\widehat I^{\,\text{anti}}_n=\frac1n\sum_{i=1}^n\frac{U_i^2+(1-U_i)^2}{2}.
$$

Because $U^2$ is increasing in $U$, the two terms in each pair err in opposite directions and partly cancel. Repeat each estimator 200 times with $n=100$ and compare the SD of the estimates with the value you already obtained in Part A2.

In [ ]:
ordinary_rng = np.random.default_rng(211)
antithetic_rng = np.random.default_rng(211)

# Generate 200 ordinary estimates of E[U**2], each from n = 100 uniforms.
ordinary_estimates = TODO

# Generate 200 antithetic estimates using (U**2 + (1-U)**2) / 2.
antithetic_estimates = TODO

# Provided comparison table -- do not edit.
variance_reduction_table = pd.DataFrame({
    "Estimator": ["Ordinary Monte Carlo", "Antithetic variables"],
    "Mean of estimates": [np.mean(ordinary_estimates), np.mean(antithetic_estimates)],
    "SD of estimates": [
        np.std(ordinary_estimates, ddof=1),
        np.std(antithetic_estimates, ddof=1),
    ],
})
variance_reduction_table.round(6)